In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters  import RecursiveCharacterTextSplitter

c:\Users\u501298\AppData\Local\miniconda3\envs\RAG_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def read_pdf_directory(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            
            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = read_pdf_directory("../data/pdf")

Found 3 PDF files to process

Processing: app - How Do Visual Explanations Foster End Users' Appropriate Trust in Machine Learning_.pdf
  Loaded 14 pages

Processing: Mrudula_Ingale_Thesis_Final_Draft.pdf
  Loaded 95 pages

Processing: us - Measures for explainable AI- Explanation goodness, user satisfaction, mental models, curiosity, trust, and human-AI performance.pdf
  Loaded 15 pages

Total documents loaded: 124


In [3]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

In [4]:
all_pdf_documents

[Document(metadata={'producer': 'iText 4.2.0 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-02-04T06:01:59-08:00', 'moddate': '2026-02-04T06:01:59-08:00', 'source': "..\\data\\pdf\\app - How Do Visual Explanations Foster End Users' Appropriate Trust in Machine Learning_.pdf", 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content=". \n. \nLatest updates: h\ue03cps://dl.acm.org/doi/10.1145/3377325.3377480\n. \n. \nRESEARCH-ARTICLE\nHow do visual explanations foster end users' appropriate trust in\nmachine learning?\nFUMENG YANG, Brown University, Providence, RI, United States\n. \nZHUANYI HUANG, Pacific Northwest National Laboratory, Richland, WA, United States\n. \nJEAN CLARICE SCHOLTZ, Pacific Northwest National Laboratory, Richland, WA, United\nStates\n. \nDUSTIN L ARENDT, Pacific Northwest National Laboratory, Richland, WA, United States\n. \n. \n. \nOpen Access Support provided by:\n. \nPacific Northwest National Laboratory\n. \nBrown University\n. \nPDF Download\n337

In [5]:
def split_documents_to_chunks(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents_to_chunks(all_pdf_documents)
chunks

Split 124 documents into 466 chunks

Example chunk:
Content: . 
. 
Latest updates: hps://dl.acm.org/doi/10.1145/3377325.3377480
. 
. 
RESEARCH-ARTICLE
How do visual explanations foster end users' appropriate trust in
machine learning?
FUMENG YANG, Brown Univer...
Metadata: {'producer': 'iText 4.2.0 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-02-04T06:01:59-08:00', 'moddate': '2026-02-04T06:01:59-08:00', 'source': "..\\data\\pdf\\app - How Do Visual Explanations Foster End Users' Appropriate Trust in Machine Learning_.pdf", 'total_pages': 14, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'iText 4.2.0 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-02-04T06:01:59-08:00', 'moddate': '2026-02-04T06:01:59-08:00', 'source': "..\\data\\pdf\\app - How Do Visual Explanations Foster End Users' Appropriate Trust in Machine Learning_.pdf", 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content=". \n. \nLatest updates: h\ue03cps://dl.acm.org/doi/10.1145/3377325.3377480\n. \n. \nRESEARCH-ARTICLE\nHow do visual explanations foster end users' appropriate trust in\nmachine learning?\nFUMENG YANG, Brown University, Providence, RI, United States\n. \nZHUANYI HUANG, Pacific Northwest National Laboratory, Richland, WA, United States\n. \nJEAN CLARICE SCHOLTZ, Pacific Northwest National Laboratory, Richland, WA, United\nStates\n. \nDUSTIN L ARENDT, Pacific Northwest National Laboratory, Richland, WA, United States\n. \n. \n. \nOpen Access Support provided by:\n. \nPacific Northwest National Laboratory\n. \nBrown University\n. \nPDF Download\n337

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error while loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embeddingManager=EmbeddingManager()
embeddingManager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 176.09it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorStore=VectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embeddingManager.generate_embeddings(texts)

##store int he vector dtaabase
vectorStore.add_documents(chunks,embeddings)


Generating embeddings for 466 texts...


Batches: 100%|██████████| 15/15 [00:41<00:00,  2.78s/it]


Generated embeddings with shape: (466, 384)
Adding 466 documents to vector store...
Successfully added 466 documents to vector store
Total documents in collection: 466


In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorStore,embeddingManager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("what is reinforcement learning?")

Retrieving documents for query: 'what is reinforcement learning?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_60cc48d9_150',
  'content': 'receive high rewards. Thus, the objective is to learn a policy that is either mapping from states to\nactions or a probability distribution over actions, that maximizes expected cumulative rewards\n[5]. Due to its ability to maximize the cumulative reward through trial-and-error, reinforcement\nlearning is especially well-suited for tasks requiring long-term sequential decision-making.\nTherefore, reinforcement learning is applied across various domains including gaming, robotics,\n5',
  'metadata': {'moddate': '2026-01-06T18:08:40+00:00',
   'creationdate': '2026-01-06T18:08:40+00:00',
   'page': 22,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1',
   'creator': 'LaTeX with hyperref',
   'author': '',
   'page_label': '5',
   'trapped': '/False',
   'source': '..\\data\\pdf\\Mrudula_Ingale_Thesis_Final_Draft.pdf',
   'keywords': '',
   'producer': 'pdfTeX-1.40.27',
   'total_pages'

In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
import os
from dotenv import load_dotenv
load_dotenv( override=True)

print("Loaded GROQ_API_KEY:", os.getenv("GROQ_API_KEY"))

Loaded GROQ_API_KEY: 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


In [15]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [16]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


In [17]:
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [18]:
answer=rag_simple("What is reinforcement learning?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is reinforcement learning?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Reinforcement learning is a branch of machine learning that focuses on making sequential decisions by learning through repeated interaction with the environment by taking actions and receiving feedback.


In [19]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("explainable AI", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'explainable AI'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Explainable AI (XAI) refers to an AI system that produces details or reasons to make its functioning clear or easy to understand, providing meaningful insights into its decision-making processes and ensuring that the reasoning behind the output is understandable to different audiences.
Sources: [{'source': '..\\data\\pdf\\Mrudula_Ingale_Thesis_Final_Draft.pdf', 'page': 23, 'score': 0.4890863299369812, 'preview': 'Chapter 2. BACKGROUND ANDRELATEDWORK\nautonomous driving, and many more.\n2.1.2 Explainable AI and Explainable RL\nWith the increasing importance of understandability and transparency in AI models, XAI has\nattracted significant attention [6]. AI systems that are used in critical domains are not only\nex...'}, {'source': '..\\data\\pdf\\Mrudula_Ingale_Thesis_Final_Draft.pdf', 'page': 23, 'score': 0.3759140968322754, 'preview': 'nations, with various feature subsets offering various degrees of insight into the agent’s\nchoices.\n• He emphasizes that explanations should 

In [20]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("explainable AI", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'explainable AI'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.40it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Chapter 2. BACKGROUND ANDRELATEDWORK
autonomous driving, and many more.
2.1.2 Explainable AI and Explainable RL
With the increasing importance of understandability and tra

nsparency in AI models, XAI has
attracted significant attention [6]. AI systems that are used in critical domains are not only
expected to make accurate decisions but also to clearly explain those decisions. As defined by
Arrieta et al. [7], “Given an audience, an explainable artificial intelligence is one that produces
details or reasons to make its functioning clear or easy to understand.” This highlights the need
for AI systems to provide meaningful insights into their decision-making processes and ensure
that the reasoning behind the output is understandable to different audiences, whether they are
developers, domain experts, or general non-expert end-users [6].
In order to comprehend an agent’s behavior inRL environments, Vouros [5] highlights a number
of important factors.

nations, with various feature subsets offering various degrees of insight into the agent’s
choices.
• He emphasizes that explanations should take into account theuser’s beliefs, perceptions,
and comprehensiono

In [22]:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Explanation goodness and satisfaction", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Explanation goodness and satisfaction'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
perspective

.
Ehsan et al. (2019) focused on the “human-likeness”
of explanations, that is, the degree to which a machine-generated
explanation seems like the sort of thing a human might say. The
researchers had players in the game Frogger express their rationales
for their game play actions. These rationales were then evaluated
by expert game players. Another group of participants evaluated
rationales that had been selected at random and rationales that
had been deemed best by a game expert. The rationales were rated
for human-likeness, and understandability. The results showed that
participants wanted rationales that were understandable, reliable, and
of suﬃcient detail. They preferred rationales that had implications for
immediate and longer-term actions.
Explanation Satisfaction is deﬁned here as the degree to which
users feel that they suﬃciently understand the AI system or process
being explained to them. Compared to Goodness, as deﬁned above,

scales is quite diﬀerent.
• The Explanation Goo